In [ ]:
from typing import Iterator
from torch.utils.data import DataLoader, SequentialSampler, Dataset

class ExampleDataset(Dataset):
    def __init__(self, size):
        self.data = list(range(size))

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx]

dataset = ExampleDataset(100)
from torch.utils.data import RandomSampler

sampler = RandomSampler(dataset)
dataloader = DataLoader(dataset, sampler=sampler, batch_size=4, drop_last=True)



class MyDataLoader(DataLoader):
    def __init__(self, dataset, batch_size = 1, shuffle = None, sampler = None, batch_sampler = None, num_workers = 0, collate_fn = None, pin_memory = False, drop_last = False, timeout = 0, worker_init_fn = None, multiprocessing_context=None, generator=None, *, prefetch_factor = None, persistent_workers = False, pin_memory_device = ""):
        super().__init__(dataset, batch_size, shuffle, sampler, batch_sampler, num_workers, collate_fn, pin_memory, drop_last, timeout, worker_init_fn, multiprocessing_context, generator, prefetch_factor=prefetch_factor, persistent_workers=persistent_workers, pin_memory_device=pin_memory_device)


import math
import torch
from torch.utils.data import Sampler

class CustomDistributedSampler(Sampler, Iterator):
    def __init__(self, dataset, n_workers=None, rank=None, shuffle=True, seed=0):
        self.dataset_size = len(dataset)
        self.n_workers = n_workers
        self.rank = rank
        self.shuffle = shuffle
        self.seed = seed
        
        self.samples_per_worker = self.dataset_size // self.n_workers

        self.total_size = self.samples_per_worker * self.n_workers

    def __next__(self):
        return next(self.indeces_iters[self.rank])

    def __len__(self):
        return self.samples_per_worker
    
    def __iter__(self):
        return self
    
    def init_iters(self):
        if self.shuffle:
            g = torch.Generator()
            g.manual_seed(self.seed)
            indices = torch.randperm(self.dataset_size, generator=g).tolist()
        else:
            indices = list(range(self.dataset_size))

        for rank in range(self.n_workers):
            start = rank * self.samples_per_worker
            end = start + self.samples_per_worker
            worker_indeces = indices[start:end]
            
            self.indeces_iters = iter(worker_indeces)


    def set_epoch(self, epoch):
        self.seed = self.seed + epoch



In [3]:
'''Get dataset mean and std with PyTorch.'''
from __future__ import print_function

import matplotlib
matplotlib.use("pdf")
import matplotlib.pyplot as plt
import logging
from datetime import datetime
from copy import deepcopy
import re

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torch.backends.cudnn as cudnn

import torchvision
import torchvision.transforms as transforms

import numpy as np

# Data
print('==> Preparing data..')
transform_train = transforms.Compose([
    transforms.ToTensor(),
])
trainset = torchvision.datasets.CIFAR100(root='./data', train=True, download=True, transform=transform_train)


trainloader = torch.utils.data.DataLoader(trainset, batch_size=1000)
device = 'cuda' if torch.cuda.is_available() else 'cpu'

h, w = 0, 0
for batch_idx, (inputs, targets) in enumerate(trainloader):
    inputs = inputs.to(device)
    if batch_idx == 0:
        h, w = inputs.size(2), inputs.size(3)
        print(inputs.min(), inputs.max())
        chsum = inputs.sum(dim=(0, 2, 3), keepdim=True)
    else:
        chsum += inputs.sum(dim=(0, 2, 3), keepdim=True)
mean = chsum/len(trainset)/h/w
print('mean: %s' % mean.view(-1))

chsum = None
for batch_idx, (inputs, targets) in enumerate(trainloader):
    inputs = inputs.to(device)
    if batch_idx == 0:
        chsum = (inputs - mean).pow(2).sum(dim=(0, 2, 3), keepdim=True)
    else:
        chsum += (inputs - mean).pow(2).sum(dim=(0, 2, 3), keepdim=True)
std = torch.sqrt(chsum/(len(trainset) * h * w - 1))
print('std: %s' % std.view(-1))

print('Done!')

==> Preparing data..
Files already downloaded and verified
tensor(0.) tensor(1.)
mean: tensor([0.5071, 0.4865, 0.4409])
std: tensor([0.2673, 0.2564, 0.2762])
Done!


In [5]:
import json
import pandas as pd
import os
def gen_dataframe(s):
    l = []
    for folder in os.listdir(s):
        folder = os.path.join(s, folder)
        if os.path.isdir(folder):
            with open(os.path.join(folder, "result.json"), "r") as f:
                res = json.load(f)

            res.pop("config")
            with open(os.path.join(folder, "params.json"), "r") as f:
                params = json.load(f)
            
            for k, v in params.items():
                res[f"config/{k}"] = v

            l.append(res)
    df = pd.DataFrame(l)
    # df.to_pickle(os.path.join(s, "analysis_results.pkl"))
    return df
    

In [ ]:
import statsmodels.api as sm


def control_for(df, name, metric, log=False):
    
    coeffs = sm.OLS(df[metric], sm.add_constant(df[])).fit().params

    return df[metric] + coeffs[f"config/{name}"] * (df[f"config/{name}"].mean() - df[f"config/{name}"])


plt.scatter(df["config/n_local_steps"], control_for(df, "n_workers", "val_loss"))

In [ ]:
df.filter(["config/local_opt.lr", "config/local_opt.weight_decay", "val_loss"]).sort_values("val_loss")#.sort_values("val_acc", ascending=False)

In [ ]:
[2**i for i in range(6, 9)]